# DERCONNECT real-data exploration and integration preparation
Checkpoint: November 6, 2026. This notebook is an offline inspection workflow, not dashboard ingestion or fault diagnosis. Clear all outputs before committing. Samples are file prefixes, not random or complete observations.

## A — Dataset overview and provenance
Instructor Haojian Jin supplied January–May 2026 data for five meters (coverage is reported, **not yet verified**). Authorized classifications: E2451 Instructional; E3020 Mixed; E3510/E7080/E7085 Office. Do not infer named buildings or electrical relationships. The approximately 3 GB local archive is not bundled or uploaded. Raw measurements and local reports must stay outside version control.

The dashboard fixtures are simulated hourly kW with simulated weather, occupancy, forecasts, labels and detections; these do not describe this archive. The browser CSV view accepts a separate provisional `timestamp,meter_id,kw` format, not arbitrary raw registers.

Install `requirements-derconnect.txt`, set `DERCONNECT_DATA_PATH` locally, then run all cells. The utility inventories ZIP metadata and streams a bounded prefix of each CSV/TSV. Parquet is optional and only read when the complete member fits the byte budget; larger Parquet needs local metadata/row-group work. Unsupported formats appear as explicit errors. No archive extraction occurs.

In [ ]:
from pathlib import Path
import os, sys, zipfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools/inspect_derconnect.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from the repository root or notebooks directory.')
sys.path.insert(0, str(ROOT / 'tools'))
from inspect_derconnect import inspect, read_member, METERS, UNITS
path_text = os.environ.get('DERCONNECT_DATA_PATH', '')
DATA_PATH = Path(path_text).expanduser() if path_text else None
report = None
if DATA_PATH is None or not DATA_PATH.is_file():
    print('Real data unavailable. Set DERCONNECT_DATA_PATH to your local ZIP and restart/run all. No findings or plots generated.')
else:
    report = inspect(DATA_PATH, rows=2000, max_bytes=8*1024*1024)
    display(pd.Series({k:v for k,v in report.items() if k not in ['files', 'schema_variants']}))
    display(pd.DataFrame([{k:v for k,v in f.items() if k in ['name','meter','months_in_name','format','compressed_bytes','uncompressed_bytes','inspection_error']} for f in report['files']]))

## B — Schema inspection
Compare every file's bounded prefix, including failures. String ingestion preserves missing encodings; inferred numeric counts are separate from storage dtype. Expected units below come from the instructor mapping and are **not verified file metadata**. Long-form `register,value,unit` files require an adapter; column-name matching alone cannot establish their availability.

In [ ]:
if report:
    display(pd.DataFrame([{'file':f['name'], 'columns':f.get('columns'), 'storage_dtypes':f.get('dtypes'), 'timestamp_candidates':f.get('timestamp_candidates'), 'error':f.get('inspection_error')} for f in report['files']]))
    print('Schema variants:', report['schema_variants'])

## C — Timestamp and sampling analysis
ISO timestamps only: choose `--timestamp` for ambiguous candidates; epoch, split date/time and local formats require explicit parsing rules. Cadence is the modal positive interval in file order, separately per file/meter. Duplicates and backwards steps are retained as evidence. Gaps use sorted unique times and a 1.5× modal interval threshold; off-cadence uses 1% tolerance. These are diagnostics, not a verified schedule. Cross-file duplicates and full-month coverage remain untested.

Naive timestamps stay naive: do not localize without source confirmation. Campus `America/Los_Angeles` crosses DST on March 8, 2026; nonexistent local times require a documented rule. Explicit offsets identify instants but do not prove a timezone. No real-data use of the dashboard's wall-time-as-UTC shortcut.

In [ ]:
if report:
    display(pd.DataFrame([{'file':f['name'], 'meter':f['meter'], **f.get('time', {}), 'note':f.get('time_warning')} for f in report['files']]))
    expected = {f'2026-{m:02}' for m in range(1,6)}
    for meter in METERS:
        observed = {token.replace('_','-') if len(token)==7 else token[:4]+'-'+token[4:] for f in report['files'] if f['meter']==meter for token in f['months_in_name']}
        print(meter, 'missing filename tokens:', sorted(expected-observed), 'unexpected:', sorted(observed-expected))
    print('Filename-month checks are provisional; use timestamps to verify actual coverage.')

## D — Measurement availability and quality
The matrix means “column observed in a sampled schema”, not “valid data for the full period”. Unknown names and metadata remain in the schema table. Ranges, missing encodings and nonnumeric counts are bounded diagnostics; units require source confirmation. Do not treat counter energy as power.

In [ ]:
if report:
    availability = pd.DataFrame({m:{r:any(r in f.get('columns',[]) for f in report['files'] if f['meter']==m) for r in UNITS} for m in METERS})
    display(availability)
    quality = pd.DataFrame([{'meter':f['meter'], 'file':f['name'], 'column':c, 'expected_unit':UNITS.get(c), **q} for f in report['files'] for c,q in f.get('quality',{}).items()])
    display(quality)

## E — Bounded exploratory visualizations
Choose a successfully inspected file and an exact register. No automatic phase sum is calculated: phase compatibility, simultaneity, net/import conventions and register semantics need confirmation. Hour/day profiles below are retrospective sample means, not energy or complete daily totals. UTC hours for offset-aware timestamps are not campus occupancy hours. Short file prefixes rarely support weekly conclusions.

Run the selected-file cell for each meter and supported voltage/frequency/current/phase register. Compare building classes only with matched windows, cadences and confirmed units; no class comparison is valid from unmatched prefixes. Change `SELECTED_NAME` to select a file explicitly.

In [ ]:
SELECTED_NAME = None
REGISTER = 'Active_Power_A'
frame = None
if report:
    candidates = [f for f in report['files'] if 'time_column' in f and REGISTER in f.get('columns', [])]
    selected = next((f for f in candidates if f['name']==SELECTED_NAME), None) if SELECTED_NAME else next(iter(candidates), None)
    if selected:
        with zipfile.ZipFile(DATA_PATH) as archive:
            frame, limited = read_member(archive, archive.getinfo(selected['name']), rows=2000)
        print('Bounded prefix:', selected['name'], 'limited:', limited)
    else:
        print('No supported timestamp/register pair. Inspect schema and select a register/parser; no plot generated.')
if frame is not None:
    if selected.get('time', {}).get('naive') and selected.get('time', {}).get('offset_aware'):
        print('Mixed time bases: resolve before plotting.')
    else:
        t = pd.to_datetime(frame[selected['time_column']], errors='coerce', utc=bool(selected.get('time',{}).get('offset_aware')))
        v = pd.to_numeric(frame[REGISTER], errors='coerce').replace([float('inf'), -float('inf')], float('nan'))
        sample = pd.DataFrame({'time':t, 'value':v}).dropna(subset=['time']).sort_values('time')
        unit = UNITS.get(REGISTER, 'unverified unit')
        fig, axes = plt.subplots(2, 3, figsize=(15,7))
        sample.plot(x='time', y='value', ax=axes[0,0], title=f'{REGISTER}: sampled readings')
        sample['value'].plot.hist(ax=axes[0,1], title='Observed value distribution')
        sample.groupby(sample.time.dt.hour)['value'].mean().plot(ax=axes[0,2], title='Sample mean by hour (timestamp basis)')
        sample.groupby(sample.time.dt.dayofweek)['value'].mean().plot(ax=axes[1,0], title='Sample mean by weekday (Monday=0)')
        sample.set_index('time')['value'].isna().resample('h').mean().plot(ax=axes[1,1], title='Missing values among observed rows')
        phases = [c for c in ['Active_Power_A','Active_Power_B','Active_Power_C'] if c in frame]
        for c in phases:
            axes[1,2].plot(t, pd.to_numeric(frame[c], errors='coerce'), label=c)
        axes[1,2].set_title('Phase power (W expected; confirm source)')
        if phases: axes[1,2].legend()
        for ax in axes.flat: ax.set_ylabel(unit)
        axes[1,1].set_ylabel('Missing fraction; absent rows excluded')
        axes[1,2].set_ylabel('W (instructor mapping)')
        plt.tight_layout()
        plt.show()

## F — Data-quality interpretation
Use Section D's adjacent-equal counts to flag candidate flatlines, not faults. Zeros and negative values can be valid; sign conventions are unknown. Counter decreases may indicate resets, rollover, ordering or schema problems. Numeric extremes need register-specific, source-confirmed thresholds. No suspicious readings are corrected, interpolated, dropped or relabeled here. Plot parsing excludes invalid timestamps only for display; the reports retain their counts.

Before cleaning, agree missing tokens, duplicate conflicts, timezone/DST policy, unit conversions, counter semantics, permissible ranges and provenance. Full-data scans and cross-file boundary checks are subsequent work; this bounded workflow cannot establish full completeness or maximum run lengths.

## G — Integration readiness and proposed contract
Proposed versioned long table: `schema_version, meter_id, timestamp_raw, timestamp_utc, source_timezone, measurement, value, unit, quality_flags, source_file, source_row`. Preserve raw text/provenance locally and nulls explicitly. Keep meter classification in a separate authorized lookup; do not fabricate building names, subgrids, occupancy, weather, PV or labels. UTC remains null until a timezone rule is agreed. Distinguish unknown units from instructor expectations.

| Concern | Required transformation or agreement |
|---|---|
| Identity | Stable five-meter IDs; reviewed mapping to building IDs and classifications |
| Time | Confirm source timezone, cadence per register/meter, duplicate policy and DST rules |
| Measurement | Preserve original register; explicit dictionary and verified units/sign conventions |
| Aggregation | Separate measured instantaneous W from cumulative kWh; convert W→kW only after confirmation; document interval weighting, minimum coverage and boundary rules |
| Model view | Derived wide table keyed by UTC time and meter, with nulls and quality mask; preserve original cadence until resampling is approved |
| Dashboard | Adapter to aligned arrays with explicit unavailable context; existing `actual[n]`/`dq[n]` is insufficient for all raw quality flags |
| Forecast/detector | Agree forecast origin/horizon/quantiles, score timestamps, thresholds, alert flags, run hashes and half-open event intervals; arrays currently start at `detector_start` |
| Evaluation | Labels are unknown, not negative ground truth; no real-data precision/recall claims |

The current CSV preview only accepts timestamp/meter_id/kw and runs no detector. This notebook does not alter that view or the synthetic pipeline.

## H — Findings and next steps
**Verified in this implementation:** archive access is local/configurable; inventory requires no extraction; supported record reads are bounded. **No real archive was available during creation**, so no real schema, cadence, completeness, plots or electrical behavior has been verified. Instructor coverage/classifications and units are supplied context only.

After local execution, record verified observations with sample scope and source filenames in ignored `local-results/`. Review notebook outputs for sensitivity and clear them before committing. Unsupported formats, byte-limited samples, long-format registers and unrecognized timestamps require adapters before further analysis.

Questions for the anomaly team: What native cadence and minimum history do models require? Which registers/phase aggregation and sign conventions are accepted? How should gaps, quality flags and duplicate conflicts be represented? Who owns timezone/resampling and classification? What forecast horizons, raw scores, thresholds, event bounds and provenance are required? Are labels or operational confirmations available?

Potential research hypotheses include load-pattern changes, phase imbalance and sensor-quality shifts; none is a confirmed fault. Next: run locally, confirm schema/time/units with Haojian, implement a reviewed canonical adapter, then compare model requirements. Full integration is outside the November 6 checkpoint.